# Predict with Mole-GNN checkpoints and verify test-set reproducibility

This notebook:

1. Loads trained Mole-GNN checkpoints for DCN, viscosity, and flashpoint.
2. Re-runs prediction on the selected `test` split.
3. Compares re-predicted values with the saved `predictions.csv` in each model folder.

If a configured model path is missing, that row is marked as `missing_model` and skipped.

In [1]:
from __future__ import annotations

from argparse import Namespace
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "scripts_training").exists() and (p / "chemprop").exists():
            return p
    raise RuntimeError(f"Could not locate repo root from {start}")


repo_root = find_repo_root(Path.cwd())
scripts_training_dir = repo_root / "scripts_training"
if str(scripts_training_dir) not in sys.path:
    sys.path.insert(0, str(scripts_training_dir))

import test_models as tm

print(f"repo_root: {repo_root}")

repo_root: /Users/u0161682/Library/CloudStorage/OneDrive-KULeuven/Documents/ScientificOutput/CodeRepositories/GitLab/chempropmix


In [2]:
# Edit these paths as needed.
jobs = [
    {
        "name": "dcn_mix_exp_fold_00",
        "mix_csv": repo_root / "datasets/fuel_ignition_numbers/processed_data/dcn_mix_exp.csv",
        "split_csv": repo_root / "splits/fuel_ignition_numbers/dcn_mix_exp/split_definitions/mixture_combination/cross_validation_10fold/fold_00.csv",
        "model_path": repo_root / "chemprop/examples/MixtureDesign/weights/dcn_mix_exp_mixture_combination_fold_00_model.pt",
        "reference_predictions_path": repo_root / "scripts_training/outputs_requested_fraction_basis/mole/gnn/dcn_mix_exp/mixture_combination/fold_00/predictions.csv",
    },
    {
        "name": "flashpoint_mix_exp_fold_00",
        "mix_csv": repo_root / "datasets/flashpoint/processed_data/flashpoint_mix_exp.csv",
        "split_csv": repo_root / "splits/flashpoint/flashpoint_mix_exp/split_definitions/mixture_combination/cross_validation_10fold/fold_00.csv",
        "model_path": repo_root / "chemprop/examples/MixtureDesign/weights/flashpoint_mix_exp_mixture_combination_fold_00_model.pt",
        "reference_predictions_path": repo_root / "scripts_training/outputs_requested_fraction_basis/mole/gnn/flashpoint_mix_exp/mixture_combination/fold_00/predictions.csv",
    },
    {
        "name": "viscosity_mix_exp_fold_00",
        "mix_csv": repo_root / "datasets/viscosity/processed_data/viscosity_mix_exp.csv",
        "split_csv": repo_root / "splits/viscosity/viscosity_mix_exp/split_definitions/mixture_combination/cross_validation_10fold/fold_00.csv",
        "model_path": repo_root / "chemprop/examples/MixtureDesign/weights/viscosity_mix_exp_mixture_combination_fold_00_model.pt",
        "reference_predictions_path": repo_root / "scripts_training/outputs_requested_fraction_basis/mole/gnn/viscosity_mix_exp/mixture_combination/fold_00/predictions.csv",
    },
]

which_split = "test"
fraction_basis = "mole"
batch_size = 64
seed = 0
aggregation = "weightedsum"
no_mixmp = False
solute_component_index = -1
use_temperature_feature = False
temperature_col = "temperature"
max_components = None
min_fraction = 1e-4

output_dir = repo_root / "chemprop/examples/MixtureDesign/repro_checks"
output_dir.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame(jobs))

,name,mix_csv,split_csv,model_path,reference_predictions_path
0,dcn_mix_exp_fold_00,/Users/u0161682/Library/CloudStorage/OneDrive-...,/Users/u0161682/Library/CloudStorage/OneDrive-...,/Users/u0161682/Library/CloudStorage/OneDrive-...,/Users/u0161682/Library/CloudStorage/OneDrive-...
1,flashpoint_mix_exp_fold_00,/Users/u0161682/Library/CloudStorage/OneDrive-...,/Users/u0161682/Library/CloudStorage/OneDrive-...,/Users/u0161682/Library/CloudStorage/OneDrive-...,/Users/u0161682/Library/CloudStorage/OneDrive-...
2,viscosity_mix_exp_fold_00,/Users/u0161682/Library/CloudStorage/OneDrive-...,/Users/u0161682/Library/CloudStorage/OneDrive-...,/Users/u0161682/Library/CloudStorage/OneDrive-...,/Users/u0161682/Library/CloudStorage/OneDrive-...


In [3]:
rows = []

for job in jobs:
    name = job["name"]
    mix_csv = Path(job["mix_csv"])
    split_csv = Path(job["split_csv"])
    model_path = Path(job["model_path"])
    ref_path = Path(job["reference_predictions_path"])

    if not model_path.exists():
        rows.append(
            {
                "name": name,
                "status": "missing_model",
                "model_path": str(model_path),
                "reference_predictions_path": str(ref_path),
            }
        )
        print(f"[SKIP] {name}: missing model {model_path}")
        continue

    if not ref_path.exists():
        rows.append(
            {
                "name": name,
                "status": "missing_reference_predictions",
                "model_path": str(model_path),
                "reference_predictions_path": str(ref_path),
            }
        )
        print(f"[SKIP] {name}: missing reference predictions {ref_path}")
        continue

    args = Namespace(
        mix_csv=mix_csv,
        split_csv=split_csv,
        which_split=which_split,
        model_path=model_path,
        seed=seed,
        batch_size=batch_size,
        max_components=max_components,
        min_fraction=min_fraction,
        fraction_basis=fraction_basis,
        aggregation=aggregation,
        no_mixmp=no_mixmp,
        solute_component_index=solute_component_index,
        use_temperature_feature=use_temperature_feature,
        temperature_col=temperature_col,
    )

    y_true, y_pred = tm._evaluate_gnn(args)

    ref_df = pd.read_csv(ref_path)
    if "y_pred" not in ref_df.columns:
        raise KeyError(f"Expected 'y_pred' column in {ref_path}")
    ref_pred = ref_df["y_pred"].to_numpy(dtype=float).reshape(-1)

    n = min(len(y_pred), len(ref_pred))
    y_pred = y_pred[:n]
    ref_pred = ref_pred[:n]
    y_true = y_true[:n]

    diff = y_pred - ref_pred
    max_abs_diff = float(np.max(np.abs(diff))) if n > 0 else np.nan
    mean_abs_diff = float(np.mean(np.abs(diff))) if n > 0 else np.nan
    exact_match = bool(np.array_equal(y_pred, ref_pred)) if n > 0 else False
    allclose_1e12 = bool(np.allclose(y_pred, ref_pred, atol=1e-12, rtol=0.0)) if n > 0 else False
    allclose_1e8 = bool(np.allclose(y_pred, ref_pred, atol=1e-8, rtol=0.0)) if n > 0 else False

    rows.append(
        {
            "name": name,
            "status": "ok",
            "n_compared": int(n),
            "exact_match": exact_match,
            "allclose_atol_1e12": allclose_1e12,
            "allclose_atol_1e8": allclose_1e8,
            "max_abs_diff": max_abs_diff,
            "mean_abs_diff": mean_abs_diff,
            "model_path": str(model_path),
            "reference_predictions_path": str(ref_path),
        }
    )

    out_df = pd.DataFrame(
        {
            "y_true": y_true,
            "y_pred_recomputed": y_pred,
            "y_pred_reference": ref_pred,
            "delta": diff,
        }
    )
    out_df.to_csv(output_dir / f"{name}_prediction_comparison.csv", index=False)
    print(
        f"[OK] {name}: n={n}, exact={exact_match}, "
        f"allclose(1e-12)={allclose_1e12}, max_abs_diff={max_abs_diff:.3e}"
    )

summary_df = pd.DataFrame(rows)
summary_path = output_dir / "mole_gnn_test_repro_summary.csv"
summary_df.to_csv(summary_path, index=False)

print(f"\nSaved summary to: {summary_path}")
display(summary_df)

[mixmp] Auto-disabling interaction module to match checkpoint (no mixmp weights).
[mixmp] enabled=False


[OK] dcn_mix_exp_fold_00: n=46, exact=False, allclose(1e-12)=False, max_abs_diff=3.679e-06
[mixmp] Auto-disabling interaction module to match checkpoint (no mixmp weights).
[mixmp] enabled=False


[OK] flashpoint_mix_exp_fold_00: n=89, exact=False, allclose(1e-12)=False, max_abs_diff=3.429e-06
[SKIP] viscosity_mix_exp_fold_00: missing model /Users/u0161682/Library/CloudStorage/OneDrive-KULeuven/Documents/ScientificOutput/CodeRepositories/GitLab/chempropmix/chemprop/examples/MixtureDesign/weights/viscosity_mix_exp_mixture_combination_fold_00_model.pt

Saved summary to: /Users/u0161682/Library/CloudStorage/OneDrive-KULeuven/Documents/ScientificOutput/CodeRepositories/GitLab/chempropmix/chemprop/examples/MixtureDesign/repro_checks/mole_gnn_test_repro_summary.csv


,name,status,n_compared,exact_match,allclose_atol_1e12,allclose_atol_1e8,max_abs_diff,mean_abs_diff,model_path,reference_predictions_path
0,dcn_mix_exp_fold_00,ok,46.0,False,False,False,0.000004,7.804539e-07,/Users/u0161682/Library/CloudStorage/OneDrive-...,/Users/u0161682/Library/CloudStorage/OneDrive-...
1,flashpoint_mix_exp_fold_00,ok,89.0,False,False,False,0.000003,3.197577e-07,/Users/u0161682/Library/CloudStorage/OneDrive-...,/Users/u0161682/Library/CloudStorage/OneDrive-...
2,viscosity_mix_exp_fold_00,missing_model,NaN,NaN,NaN,NaN,NaN,NaN,/Users/u0161682/Library/CloudStorage/OneDrive-...,/Users/u0161682/Library/CloudStorage/OneDrive-...
